# Segundo trabajo de la asignatura: Redes Neuronales y Algoritmos Bioinspirados. 


**Universidad Nacional de Colombia**

Departamento de Ciencias de la Computación y de la Decisión

Integrantes:  

**-Juan Manuel Diaz Larrea**

**-Kevin Hair González Duarte**

**-Manuel Gutierrez Tangarife**

**-Mariana Ruiz Zamora**

**-Maria Fernanda Valencia Jimenez**



Semestre 2026-02


# Solución parte 1: Probabilidad de incumplimiento (clasificación binaria) 

**Objetivo: Modelar la probabilidad de que un cliente sea "malo", es decir, que incumpla con el pago de su crédito**

In [32]:
#Cargar librerías 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
#Lectura de datos   

#Fecha de descarga: 24/10/2026

datos= pd.read_csv("loan.csv")



C:\Users\Maria Fernanda\AppData\Local\Temp\ipykernel_4340\212864074.py:5: DtypeWarning: Columns (0: desc, 1: verification_status_joint) have mixed types. Specify dtype option on import or set low_memory=False.
  datos= pd.read_csv("loan.csv")


In [3]:
#Verificar que se leyó bien la bd

datos.head()
datos.tail()
datos.sample(6)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
560801,63204225,67555941,14000.0,14000.0,14000.0,60 months,12.69,316.33,C,C2,...,NaN,NaN,NaN,NaN,NaN,NaN,39000.0,NaN,NaN,NaN
850705,40361054,43225806,4800.0,4800.0,4800.0,36 months,14.99,166.38,C,C5,...,NaN,NaN,NaN,NaN,NaN,NaN,7600.0,NaN,NaN,NaN
542696,64027582,68458303,6475.0,6475.0,6475.0,36 months,13.18,218.74,C,C3,...,NaN,NaN,NaN,NaN,NaN,NaN,14500.0,NaN,NaN,NaN
441324,11956172,13948325,10000.0,10000.0,10000.0,60 months,15.31,239.53,C,C4,...,NaN,NaN,NaN,NaN,NaN,NaN,11600.0,NaN,NaN,NaN
869612,38952283,41735999,30000.0,30000.0,30000.0,60 months,16.49,737.38,D,D3,...,NaN,NaN,NaN,NaN,NaN,NaN,34600.0,NaN,NaN,NaN
163192,3380283,4233221,17000.0,17000.0,17000.0,60 months,13.11,387.77,B,B4,...,NaN,NaN,NaN,NaN,NaN,NaN,11900.0,NaN,NaN,NaN


In [4]:
#Tamaño de la bd

datos.shape

#Hay 887,379 filas y 74 columnas 

(887379, 74)

In [5]:
#Cuáles son las columnas de la bd y qué representan

datos.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose',
       'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
       'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq',
       'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'initial_list_status', 'out_prncp',
       'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
       'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med',
       'mths_since_last_major_derog', 'policy_code', 'application_type',
       'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    

Según el sitio web: https://www.kaggle.com/datasets/ranadeep/credit-risk-dataset/data el diccionario de datos es:

### Identificación y tipo de solicitud

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>id</code></td><td style="text-align:left">Identificador único del listado del préstamo, asignado por LC.</td></tr>
<tr><td style="text-align:left"><code>member_id</code></td><td style="text-align:left">Identificador único del miembro prestatario, asignado por LC.</td></tr>
<tr><td style="text-align:left"><code>url</code></td><td style="text-align:left">URL de la página de LC con los datos del listado.</td></tr>
<tr><td style="text-align:left"><code>policy_code</code></td><td style="text-align:left">Productos disponibles al público: <code>policy_code</code>&nbsp;=&nbsp;1.<br> Productos nuevos no disponibles al público: <code>policy_code</code>&nbsp;=&nbsp;2.</td></tr>
<tr><td style="text-align:left"><code>initial_list_status</code></td><td style="text-align:left">Estado inicial de publicación del préstamo. Los valores posibles son W y F.</td></tr>
<tr><td style="text-align:left"><code>application_type</code></td><td style="text-align:left">Indica si el préstamo corresponde a una solicitud individual o a una solicitud conjunta con dos coprestatarios.</td></tr>
</tbody>
</table>

### Condiciones del préstamo

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>loan_amnt</code></td><td style="text-align:left">Monto del préstamo solicitado por el prestatario, tal como figura en el listado. Si en algún momento el departamento de crédito reduce el monto del préstamo, dicha reducción se verá reflejada en este valor.</td></tr>
<tr><td style="text-align:left"><code>funded_amnt</code></td><td style="text-align:left">Monto total comprometido (financiado) para ese préstamo en ese momento.</td></tr>
<tr><td style="text-align:left"><code>funded_amnt_inv</code></td><td style="text-align:left">Monto total comprometido por los inversionistas para ese préstamo en ese momento.</td></tr>
<tr><td style="text-align:left"><code>term</code></td><td style="text-align:left">Número de pagos del préstamo. Los valores están expresados en meses y pueden ser 36 o 60.</td></tr>
<tr><td style="text-align:left"><code>int_rate</code></td><td style="text-align:left">Tasa de interés del préstamo.</td></tr>
<tr><td style="text-align:left"><code>installment</code></td><td style="text-align:left">Cuota mensual que debe pagar el prestatario si el préstamo se origina.</td></tr>
<tr><td style="text-align:left"><code>grade</code></td><td style="text-align:left">Calificación (<i>grade</i>) del préstamo asignada por LC.</td></tr>
<tr><td style="text-align:left"><code>sub_grade</code></td><td style="text-align:left">Subcalificación (<i>subgrade</i>) del préstamo asignada por LC.</td></tr>
<tr><td style="text-align:left"><code>issue_d</code></td><td style="text-align:left">Mes en que se desembolsó (financió) el préstamo.</td></tr>
<tr><td style="text-align:left"><code>purpose</code></td><td style="text-align:left">Categoría indicada por el prestatario como finalidad del préstamo solicitado.</td></tr>
<tr><td style="text-align:left"><code>title</code></td><td style="text-align:left">Título del préstamo indicado por el prestatario.</td></tr>
<tr><td style="text-align:left"><code>desc</code></td><td style="text-align:left">Descripción del préstamo proporcionada por el prestatario.</td></tr>
<tr><td style="text-align:left"><code>pymnt_plan</code></td><td style="text-align:left">Indica si se estableció un plan de pagos para el préstamo.</td></tr>
<tr><td style="text-align:left"><code>loan_status</code></td><td style="text-align:left">Estado actual del préstamo.</td></tr>
</tbody>
</table>

###  Perfil del prestatario

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>addr_state</code></td><td style="text-align:left">Estado (de EE.&nbsp;UU.) indicado por el prestatario en la solicitud del préstamo.</td></tr>
<tr><td style="text-align:left"><code>zip_code</code></td><td style="text-align:left">Los tres primeros dígitos del código postal indicado por el prestatario en la solicitud del préstamo.</td></tr>
<tr><td style="text-align:left"><code>emp_title</code></td><td style="text-align:left">Cargo (título del puesto de trabajo) indicado por el prestatario al solicitar el préstamo.<sup>*</sup></td></tr>
<tr><td style="text-align:left"><code>emp_length</code></td><td style="text-align:left">Antigüedad laboral en años. Los valores posibles van de 0 a 10, donde 0 significa menos de un año y 10 significa diez años o más.</td></tr>
<tr><td style="text-align:left"><code>home_ownership</code></td><td style="text-align:left">Situación de vivienda del prestatario informada durante el registro. Los valores son: RENT (arriendo), OWN (vivienda propia), MORTGAGE (vivienda con hipoteca) y OTHER (otra).</td></tr>
<tr><td style="text-align:left"><code>annual_inc</code></td><td style="text-align:left">Ingreso anual autodeclarado por el prestatario durante el registro.</td></tr>
<tr><td style="text-align:left"><code>annual_inc_joint</code></td><td style="text-align:left">Ingreso anual combinado, autodeclarado por los coprestatarios durante el registro.</td></tr>
<tr><td style="text-align:left"><code>dti</code></td><td style="text-align:left">Razón (relación deuda-ingreso) calculada como los pagos mensuales totales del prestatario por el total de sus obligaciones de deuda, excluyendo la hipoteca y el préstamo solicitado a LC, dividida entre el ingreso mensual autodeclarado del prestatario.</td></tr>
<tr><td style="text-align:left"><code>dti_joint</code></td><td style="text-align:left">Razón (relación deuda-ingreso conjunta) calculada como los pagos mensuales totales de los coprestatarios por el total de sus obligaciones de deuda, excluyendo las hipotecas y el préstamo solicitado a LC, dividida entre el ingreso mensual combinado autodeclarado de los coprestatarios.</td></tr>
<tr><td style="text-align:left"><code>is_inc_v</code></td><td style="text-align:left">Indica si LC verificó el ingreso, si no fue verificado o si se verificó la fuente del ingreso.</td></tr>
<tr><td style="text-align:left"><code>verified_status_joint</code></td><td style="text-align:left">Indica si LC verificó el ingreso conjunto de los coprestatarios, si no fue verificado o si se verificó la fuente del ingreso.</td></tr>
</tbody>
</table>

### Historial crediticio general

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>fico_range_low</code></td><td style="text-align:left">Límite inferior del rango al que pertenece el puntaje FICO del prestatario en el momento de originación del préstamo.</td></tr>
<tr><td style="text-align:left"><code>fico_range_high</code></td><td style="text-align:left">Límite superior del rango al que pertenece el puntaje FICO del prestatario en el momento de originación del préstamo.</td></tr>
<tr><td style="text-align:left"><code>last_fico_range_low</code></td><td style="text-align:left">Límite inferior del rango al que pertenece el último puntaje FICO consultado del prestatario.</td></tr>
<tr><td style="text-align:left"><code>last_fico_range_high</code></td><td style="text-align:left">Límite superior del rango al que pertenece el último puntaje FICO consultado del prestatario.</td></tr>
<tr><td style="text-align:left"><code>earliest_cr_line</code></td><td style="text-align:left">Mes en que se abrió la línea de crédito más antigua reportada del prestatario.</td></tr>
<tr><td style="text-align:left"><code>last_credit_pull_d</code></td><td style="text-align:left">Mes más reciente en que LC consultó el reporte de crédito para este préstamo.</td></tr>
<tr><td style="text-align:left"><code>inq_last_6mths</code></td><td style="text-align:left">Número de consultas crediticias en los últimos 6 meses (excluyendo las consultas por crédito de vehículo e hipotecario).</td></tr>
<tr><td style="text-align:left"><code>inq_last_12m</code></td><td style="text-align:left">Número de consultas crediticias en los últimos 12 meses.</td></tr>
<tr><td style="text-align:left"><code>inq_fi</code></td><td style="text-align:left">Número de consultas crediticias de finanzas personales.</td></tr>
<tr><td style="text-align:left"><code>open_acc</code></td><td style="text-align:left">Número de líneas de crédito abiertas en el historial crediticio del prestatario.</td></tr>
<tr><td style="text-align:left"><code>total_acc</code></td><td style="text-align:left">Número total de líneas de crédito que figuran actualmente en el historial crediticio del prestatario.</td></tr>
<tr><td style="text-align:left"><code>pub_rec</code></td><td style="text-align:left">Número de registros públicos desfavorables (<i>derogatory</i>).</td></tr>
<tr><td style="text-align:left"><code>revol_bal</code></td><td style="text-align:left">Saldo total del crédito rotativo.</td></tr>
<tr><td style="text-align:left"><code>revol_util</code></td><td style="text-align:left">Tasa de utilización de las líneas rotativas, es decir, la proporción del crédito que el prestatario está usando con respecto al total del crédito rotativo disponible.</td></tr>
<tr><td style="text-align:left"><code>total_rev_hi_lim</code></td><td style="text-align:left">Total del crédito máximo/límite de crédito de las cuentas rotativas.</td></tr>
<tr><td style="text-align:left"><code>tot_cur_bal</code></td><td style="text-align:left">Saldo actual total de todas las cuentas.</td></tr>
<tr><td style="text-align:left"><code>total_cu_tl</code></td><td style="text-align:left">Número de líneas de crédito de financiamiento (<i>finance trades</i>).</td></tr>
<tr><td style="text-align:left"><code>all_util</code></td><td style="text-align:left">Razón entre el saldo y el límite de crédito de todas las líneas de crédito.</td></tr>
<tr><td style="text-align:left"><code>max_bal_bc</code></td><td style="text-align:left">Saldo actual máximo adeudado en todas las cuentas rotativas.</td></tr>
</tbody>
</table>

### Cuentas a cuotas y rotativas

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>open_acc_6m</code></td><td style="text-align:left">Número de líneas de crédito (<i>trades</i>) abiertas en los últimos 6 meses.</td></tr>
<tr><td style="text-align:left"><code>open_il_6m</code></td><td style="text-align:left">Número de líneas de crédito a cuotas (<i>installment</i>) actualmente activas.</td></tr>
<tr><td style="text-align:left"><code>open_il_12m</code></td><td style="text-align:left">Número de cuentas a cuotas abiertas en los últimos 12 meses.</td></tr>
<tr><td style="text-align:left"><code>open_il_24m</code></td><td style="text-align:left">Número de cuentas a cuotas abiertas en los últimos 24 meses.</td></tr>
<tr><td style="text-align:left"><code>mths_since_rcnt_il</code></td><td style="text-align:left">Meses transcurridos desde la apertura de la cuenta a cuotas más reciente.</td></tr>
<tr><td style="text-align:left"><code>total_bal_il</code></td><td style="text-align:left">Saldo actual total de todas las cuentas a cuotas.</td></tr>
<tr><td style="text-align:left"><code>il_util</code></td><td style="text-align:left">Razón entre el saldo actual total y el crédito máximo/límite de crédito de todas las cuentas a cuotas.</td></tr>
<tr><td style="text-align:left"><code>open_rv_12m</code></td><td style="text-align:left">Número de líneas de crédito rotativas abiertas en los últimos 12 meses.</td></tr>
<tr><td style="text-align:left"><code>open_rv_24m</code></td><td style="text-align:left">Número de líneas de crédito rotativas abiertas en los últimos 24 meses.</td></tr>
</tbody>
</table>

###  Mora y cobranza

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>delinq_2yrs</code></td><td style="text-align:left">Número de episodios de mora de 30 o más días de vencimiento registrados en el historial crediticio del prestatario durante los últimos 2 años.</td></tr>
<tr><td style="text-align:left"><code>mths_since_last_delinq</code></td><td style="text-align:left">Número de meses transcurridos desde la última mora del prestatario.</td></tr>
<tr><td style="text-align:left"><code>mths_since_last_major_derog</code></td><td style="text-align:left">Meses transcurridos desde la calificación más reciente de 90 días de mora o peor.</td></tr>
<tr><td style="text-align:left"><code>mths_since_last_record</code></td><td style="text-align:left">Número de meses transcurridos desde el último registro público.</td></tr>
<tr><td style="text-align:left"><code>acc_now_delinq</code></td><td style="text-align:left">Número de cuentas en las que el prestatario se encuentra actualmente en mora.</td></tr>
<tr><td style="text-align:left"><code>collections_12_mths_ex_med</code></td><td style="text-align:left">Número de cobranzas en los últimos 12 meses, excluyendo las cobranzas de origen médico.</td></tr>
<tr><td style="text-align:left"><code>tot_coll_amt</code></td><td style="text-align:left">Monto total histórico adeudado en cobranza.</td></tr>
</tbody>
</table>

### Pagos y recuperaciones

<table style="width:100%">
<colgroup><col style="width:26%"><col style="width:74%"></colgroup>
<thead><tr><th style="text-align:left">Variable</th><th style="text-align:left">Descripción</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><code>out_prncp</code></td><td style="text-align:left">Capital pendiente por pagar sobre el monto total financiado.</td></tr>
<tr><td style="text-align:left"><code>out_prncp_inv</code></td><td style="text-align:left">Capital pendiente por pagar sobre la porción del monto total financiado por los inversionistas.</td></tr>
<tr><td style="text-align:left"><code>total_pymnt</code></td><td style="text-align:left">Pagos recibidos a la fecha sobre el monto total financiado.</td></tr>
<tr><td style="text-align:left"><code>total_pymnt_inv</code></td><td style="text-align:left">Pagos recibidos a la fecha sobre la porción del monto total financiado por los inversionistas.</td></tr>
<tr><td style="text-align:left"><code>total_rec_prncp</code></td><td style="text-align:left">Capital recibido a la fecha.</td></tr>
<tr><td style="text-align:left"><code>total_rec_int</code></td><td style="text-align:left">Intereses recibidos a la fecha.</td></tr>
<tr><td style="text-align:left"><code>total_rec_late_fee</code></td><td style="text-align:left">Cargos por pago tardío recibidos a la fecha.</td></tr>
<tr><td style="text-align:left"><code>recoveries</code></td><td style="text-align:left">Recuperación bruta posterior al castigo de cartera (<i>charge off</i>).</td></tr>
<tr><td style="text-align:left"><code>collection_recovery_fee</code></td><td style="text-align:left">Comisión de cobranza posterior al castigo de cartera (<i>charge off</i>).</td></tr>
<tr><td style="text-align:left"><code>last_pymnt_d</code></td><td style="text-align:left">Mes en que se recibió el último pago.</td></tr>
<tr><td style="text-align:left"><code>last_pymnt_amnt</code></td><td style="text-align:left">Monto total del último pago recibido.</td></tr>
<tr><td style="text-align:left"><code>next_pymnt_d</code></td><td style="text-align:left">Fecha programada del próximo pago.</td></tr>
</tbody>
</table>

<sup>*</sup> El cargo (<i>Employer Title</i>) reemplaza al nombre del empleador en todos los préstamos listados después del 23 de septiembre de 2013.

In [6]:
#Verificar si hay valores duplicados en la bd

print("Número de filas antes de eliminar registros duplicados:", datos.shape[0])
datos = datos.drop_duplicates()
print("Número de filas después de eliminar registros duplicados:", datos.shape[0])

#No hay registros duplicados en la bd 

Número de filas antes de eliminar registros duplicados: 887379
Número de filas después de eliminar registros duplicados: 887379


In [8]:
#información de la bd

datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 887379 entries, 0 to 887378
Data columns (total 74 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           887379 non-null  int64  
 1   member_id                    887379 non-null  int64  
 2   loan_amnt                    887379 non-null  float64
 3   funded_amnt                  887379 non-null  float64
 4   funded_amnt_inv              887379 non-null  float64
 5   term                         887379 non-null  str    
 6   int_rate                     887379 non-null  float64
 7   installment                  887379 non-null  float64
 8   grade                        887379 non-null  str    
 9   sub_grade                    887379 non-null  str    
 10  emp_title                    835917 non-null  str    
 11  emp_length                   842554 non-null  str    
 12  home_ownership               887379 non-null  str    
 13  annual_inc

In [9]:
#Resumen de las variables numéricas de la bd

datos.describe()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
count,8.873790e+05,8.873790e+05,887379.000000,887379.000000,887379.000000,887379.000000,887379.000000,8.873750e+05,887379.000000,887350.000000,...,21372.000000,18617.000000,21372.000000,21372.000000,21372.000000,21372.000000,8.171030e+05,21372.000000,21372.000000,21372.000000
mean,3.246513e+07,3.500182e+07,14755.264605,14741.877625,14702.464383,13.246740,436.717127,7.502759e+04,18.157039,0.314442,...,36552.811389,71.535317,1.389060,2.975482,5887.979740,60.831939,3.206862e+04,0.943945,1.537058,1.977307
std,2.282734e+07,2.411335e+07,8435.455601,8429.897657,8442.106732,4.381867,244.186593,6.469830e+04,17.190626,0.862244,...,43103.833619,23.022862,1.520129,2.631886,5284.701239,20.013254,3.749826e+04,1.446872,2.723339,2.874067
min,5.473400e+04,7.047300e+04,500.000000,500.000000,0.000000,5.320000,15.670000,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,-4.000000
25%,9.206643e+06,1.087713e+07,8000.000000,8000.000000,8000.000000,9.990000,260.705000,4.500000e+04,11.910000,0.000000,...,10251.500000,58.600000,0.000000,1.000000,2411.000000,47.700000,1.390000e+04,0.000000,0.000000,0.000000
50%,3.443327e+07,3.709528e+07,13000.000000,13000.000000,13000.000000,12.990000,382.550000,6.500000e+04,17.650000,0.000000,...,24684.500000,74.900000,1.000000,2.000000,4483.000000,61.900000,2.370000e+04,0.000000,0.000000,2.000000
75%,5.490814e+07,5.847135e+07,20000.000000,20000.000000,20000.000000,16.200000,572.600000,9.000000e+04,23.950000,0.000000,...,47858.000000,87.600000,2.000000,4.000000,7771.500000,75.200000,3.980000e+04,1.000000,2.000000,3.000000
max,6.861706e+07,7.354484e+07,35000.000000,35000.000000,35000.000000,28.990000,1445.460000,9.500000e+06,9999.000000,39.000000,...,878459.000000,223.300000,22.000000,43.000000,127305.000000,151.400000,9.999999e+06,17.000000,35.000000,32.000000


In [10]:
#Resumen de las variables categóricas de la bd

datos.describe(include="object")

C:\Users\Maria Fernanda\AppData\Local\Temp\ipykernel_4340\3657833786.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  datos.describe(include="object")


,term,grade,sub_grade,emp_title,emp_length,home_ownership,verification_status,issue_d,loan_status,pymnt_plan,...,title,zip_code,addr_state,earliest_cr_line,initial_list_status,last_pymnt_d,next_pymnt_d,last_credit_pull_d,application_type,verification_status_joint
count,887379,887379,887379,835917,842554,887379,887379,887379,887379,887379,...,887226,887379,887379,887350,887379,869720,634408,887326,887379,511
unique,2,7,35,299271,11,6,3,103,10,2,...,63143,935,51,697,2,98,100,103,2,3
top,36 months,B,B3,Teacher,10+ years,MORTGAGE,Source Verified,Oct-2015,Current,n,...,Debt consolidation,945xx,CA,Aug-2001,f,Jan-2016,Feb-2016,Jan-2016,INDIVIDUAL,Not Verified
freq,621125,254535,56323,13469,291569,443557,329558,48631,601779,887369,...,414001,9770,129517,6659,456848,470150,553406,730574,886868,283


In [7]:
#Verificar valores nulos

datos.isna().sum()

id                       0
member_id                0
loan_amnt                0
funded_amnt              0
funded_amnt_inv          0
                     ...  
all_util            866007
total_rev_hi_lim     70276
inq_fi              866007
total_cu_tl         866007
inq_last_12m        866007
Length: 74, dtype: int64

In [11]:
#Porcentaje de valores nulos por columna

datos.isnull().sum() / len(datos) * 100

id                   0.000000
member_id            0.000000
loan_amnt            0.000000
funded_amnt          0.000000
funded_amnt_inv      0.000000
                      ...    
all_util            97.591559
total_rev_hi_lim     7.919502
inq_fi              97.591559
total_cu_tl         97.591559
inq_last_12m        97.591559
Length: 74, dtype: float64

In [16]:
#Eliminación de columnas con más del 90% de valores nulos

umbral = 90 
columnas_nulos = datos.isnull().sum() / len(datos) * 100

cols_eliminar = columnas_nulos[columnas_nulos > umbral].index.tolist()

print(f"Se eliminan {len(cols_eliminar)} columnas:")
print(columnas_nulos[cols_eliminar].round(2).sort_values(ascending=False))

datos = datos.drop(columns=cols_eliminar)




Se eliminan 17 columnas:
annual_inc_joint             99.94
dti_joint                    99.94
verification_status_joint    99.94
il_util                      97.90
mths_since_rcnt_il           97.65
open_il_6m                   97.59
open_il_12m                  97.59
open_il_24m                  97.59
open_acc_6m                  97.59
total_bal_il                 97.59
open_rv_12m                  97.59
open_rv_24m                  97.59
max_bal_bc                   97.59
all_util                     97.59
inq_fi                       97.59
total_cu_tl                  97.59
inq_last_12m                 97.59
dtype: float64


# Selección de variables 

Teniendo en cuenta el objetivo del primer punto, las variables que se van a considerar son:

Variable objetivo: loan_status

Variables a usar: issue_d (particion), loan_amnt, term, emp_length, home_ownership, annual_inc, purpose, dti, delinq_2yrs, earliest_cr_line, inq_last_6mths, mths_since_last_delinq, mths_since_last_record, open_acc, pub_rec, revol_bal, revol_util, total_acc



In [21]:
#Selección de variables/columnas

columnas = ["loan_status", "issue_d",
            "loan_amnt", "term", "emp_length", "home_ownership", "annual_inc", "purpose",
            "dti", "delinq_2yrs", "earliest_cr_line", "inq_last_6mths",
            "mths_since_last_delinq", "mths_since_last_record", "open_acc",
            "pub_rec", "revol_bal", "revol_util", "total_acc"]

faltan = set(columnas) - set(datos.columns)
assert not faltan, f"Columnas ausentes: {faltan}"

df = datos[columnas].copy()

# Codificación de la variable *loan_status*

Se hará la codificación pedida por el profesor a la variable *loan_status* 

In [37]:
mapa = {
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1,
    "Late (31-120 days)": 1,
    "Does not meet the credit policy. Status:Charged Off": 1,
}
estados_na = {"Current", "Issued", "In Grace Period", "Late (16-30 days)"}

# 1) Limpieza de texto y verificación de categorías
assert df["loan_status"].notna().all(), "Hay loan_status vacíos"
df["loan_status"] = df["loan_status"].str.strip()

desconocidos = set(df["loan_status"].unique()) - set(mapa) - estados_na
assert not desconocidos, f"Categorías no contempladas en el enunciado: {desconocidos}"

# 2) Variable binaria: 1 = malo, 0 = bueno, NaN = estados excluidos
df["y"] = df["loan_status"].map(mapa)

# 3) Conteos para el reporte (antes de excluir)
conteo = (df["loan_status"].value_counts()
          .rename_axis("loan_status").reset_index(name="n"))
conteo["codificacion"] = conteo["loan_status"].map(mapa)   # NaN = excluido
print(conteo)
print("Registros con y = NA (excluidos):", df["y"].isna().sum())

# 4) Excluir los NA y dejar y como entero
df = df.dropna(subset=["y"]).copy()
df["y"] = df["y"].astype(int)

# 5) Resumen del conjunto de modelamiento
print(df["y"].value_counts())
print(f"Tasa de malos: {df['y'].mean():.2%}")

# 6) Evitar usar loan_status como predictor por accidente 
df = df.drop(columns="loan_status")

                                         loan_status       n  codificacion
0                                            Current  601779           NaN
1                                         Fully Paid  207723           0.0
2                                        Charged Off   45248           1.0
3                                 Late (31-120 days)   11591           1.0
4                                             Issued    8460           NaN
5                                    In Grace Period    6253           NaN
6                                  Late (16-30 days)    2357           NaN
7  Does not meet the credit policy. Status:Fully ...    1988           0.0
8                                            Default    1219           1.0
9  Does not meet the credit policy. Status:Charge...     761           1.0
Registros con y = NA (excluidos): 618849
y
0    209711
1     58819
Name: count, dtype: int64
Tasa de malos: 21.90%


## Decisión de los cortes

In [ ]:
df["issue_date"] = pd.to_datetime(df["issue_d"], format='mixed')
df["anio"] = df["issue_date"].dt.year

tabla = df.groupby("anio")["y"].agg(etiquetados="size", malos="sum", tasa_malos="mean")
print(tabla)

      etiquetados  malos  tasa_malos
anio                                
2007          603    158    0.262023
2008         2393    496    0.207271
2009         5281    723    0.136906
2010        12528   1759    0.140405
2011        19729   3259    0.165188
2012        49813   8334    0.167306
2013        73194  16798    0.229500
2014        74154  19441    0.262171
2015        30835   7851    0.254613


In [47]:
total_anio = pd.to_datetime(datos["issue_d"], format="mixed").dt.year.value_counts().sort_index()
tabla["total"] = total_anio
tabla["pct_etiquetados"] = (tabla["etiquetados"] / tabla["total"]).round(3)
print(tabla)

      etiquetados  malos  tasa_malos   total  pct_etiquetados
anio                                                         
2007          603    158    0.262023     603            1.000
2008         2393    496    0.207271    2393            1.000
2009         5281    723    0.136906    5281            1.000
2010        12528   1759    0.140405   12537            0.999
2011        19729   3259    0.165188   21721            0.908
2012        49813   8334    0.167306   53367            0.933
2013        73194  16798    0.229500  134755            0.543
2014        74154  19441    0.262171  235628            0.315
2015        30835   7851    0.254613  421094            0.073


## Datos de entrenamiento, validación y prueba 

In [49]:
def asigna(anio):
    if anio <= 2010: return "train"
    if anio == 2011: return "valid"
    if anio == 2012: return "test"
    return "fuera_censurado"      # 2013-2015

df["particion"] = df["anio"].map(asigna)

resumen = df.groupby("particion")["y"].agg(n="size", malos="sum", tasa="mean")
print(resumen)

# Verificaciones: deben coincidir con la tabla
assert resumen.loc["train", "n"] == 20805
assert resumen.loc["valid", "n"] == 19729
assert resumen.loc["test", "n"] == 49813
assert resumen["n"].sum() == len(df)            

# Conjuntos separados
train = df[df["particion"] == "train"].copy()
valid = df[df["particion"] == "valid"].copy()
test  = df[df["particion"] == "test"].copy()

                      n  malos      tasa
particion                               
fuera_censurado  178183  44090  0.247442
test              49813   8334  0.167306
train             20805   3136  0.150733
valid             19729   3259  0.165188


## Análisis descriptivo

### Análisis descriptivo de las variables numéricas

In [51]:
train.info()

<class 'pandas.DataFrame'>
Index: 20805 entries, 21494 to 42534
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   issue_d                 20805 non-null  str           
 1   loan_amnt               20805 non-null  float64       
 2   term                    20805 non-null  str           
 3   emp_length              20453 non-null  str           
 4   home_ownership          20805 non-null  str           
 5   annual_inc              20801 non-null  float64       
 6   purpose                 20805 non-null  str           
 7   dti                     20805 non-null  float64       
 8   delinq_2yrs             20776 non-null  float64       
 9   earliest_cr_line        20776 non-null  str           
 10  inq_last_6mths          20776 non-null  float64       
 11  mths_since_last_delinq  8266 non-null   float64       
 12  mths_since_last_record  2505 non-null   float64       
 13